# Word Count Example - PySpark RDD

This notebook demonstrates basic RDD operations using word count example.

In [ ]:
# Install and setup Java (for Google Colab)
import os

def install_java():
    !apt-get install -y openjdk-8-jdk-headless -qq > /dev/null
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
    !java -version

install_java()

In [ ]:
# Install PySpark
!pip install pyspark

In [ ]:
# Import and create Spark Session
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName('Word Count Example') \
    .getOrCreate()

sc = spark.sparkContext
print(f"Spark Version: {spark.version}")

## Basic Word Count Example

In [ ]:
# Create sample text data
text_data = [
    "Apache Spark is a unified analytics engine",
    "Spark is fast and general purpose cluster computing",
    "Spark provides high level APIs in Java Python and Scala",
    "Spark is built on top of Hadoop MapReduce"
]

# Create RDD from text data
text_rdd = sc.parallelize(text_data)
print(f"Number of partitions: {text_rdd.getNumPartitions()}")

In [ ]:
# Word count using RDD transformations
word_counts = text_rdd \
    .flatMap(lambda line: line.lower().split()) \
    .map(lambda word: (word, 1)) \
    .reduceByKey(lambda a, b: a + b) \
    .sortBy(lambda x: x[1], ascending=False)

# Collect and display results
results = word_counts.collect()
print("\nWord Count Results:")
print("-" * 30)
for word, count in results[:10]:
    print(f"{word:20s} : {count}")

## Reading from File and Word Count

In [ ]:
# Create a sample text file
sample_text = """Apache Spark is a unified analytics engine for large-scale data processing.
Spark provides high-level APIs in Java, Scala, Python and R.
Spark is built on the foundation of RDDs and DataFrames.
Spark can run on Hadoop, Apache Mesos, Kubernetes, standalone, or in the cloud."""

# Write to file
with open('/tmp/sample_text.txt', 'w') as f:
    f.write(sample_text)

print("Sample file created at /tmp/sample_text.txt")

In [ ]:
# Read from file and perform word count
file_rdd = sc.textFile('/tmp/sample_text.txt')

word_count_from_file = file_rdd \
    .flatMap(lambda line: line.lower().split()) \
    .filter(lambda word: len(word) > 3) \
    .map(lambda word: (word, 1)) \
    .reduceByKey(lambda a, b: a + b) \
    .sortBy(lambda x: x[1], ascending=False)

print("Top 10 words (length > 3):")
print("-" * 30)
for word, count in word_count_from_file.take(10):
    print(f"{word:20s} : {count}")

## RDD Partitioning and Performance

In [ ]:
# Check default parallelism
print(f"Default Parallelism: {sc.defaultParallelism}")

# Create RDD with specific number of partitions
rdd_with_partitions = sc.parallelize(range(1, 1001), 8)
print(f"Number of partitions: {rdd_with_partitions.getNumPartitions()}")

# Repartition example
rdd_repartitioned = rdd_with_partitions.repartition(4)
print(f"After repartition: {rdd_repartitioned.getNumPartitions()}")

# Coalesce example (reduces partitions without shuffle)
rdd_coalesced = rdd_with_partitions.coalesce(2)
print(f"After coalesce: {rdd_coalesced.getNumPartitions()}")

## Save Results

In [ ]:
# Save word count results to file
output_path = '/tmp/word_count_output'

# Remove output directory if exists
!rm -rf {output_path}

# Save as text file
word_counts.coalesce(1).saveAsTextFile(output_path)

print(f"Results saved to {output_path}")
print("\nOutput files:")
!ls -lh {output_path}

In [ ]:
# Stop Spark Session
spark.stop()